### Tool calling for AI agents

#### Basic Agentic AI idea
> ##### We can have 2 forms:
- The llm has tools and when the llm call for those tools other llm call are initiated in those tools
- The llm make the to do list and then perform the task as the agent.

In [1]:
import os
import json
import gradio as gr
from IPython.display import Markdown,display
from dotenv import load_dotenv
from litellm import completion
load_dotenv(override=True)

c:\Users\Sushant\Documents\IntelligenceStack\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
gemini_api_key = os.getenv("GOOGLE_API_KEY")
cerebras_api_key = os.getenv("CEREBRAS_API_KEY")
groq_api_key = os.getenv("GROQ_API_KEY")

In [ ]:
system_message = """
You are a helpful assistant for an Restraunt desk help called ResDeskHelp.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
only answer the price of the dish if you know them if don't please say so
"""

In [12]:
res_menu = {"dalchawal":899, "butterpaneer":6666, "bhindi":789, "pulao":842}
def dish_rate(dish):
    print(f"Menu price check called for {dish}")
    price = res_menu.get(dish.lower(),"dish not listed")
    return f'The price for {dish} is {price}'


In [5]:
dish_rate("dalchawal")

Menu price check called for dalchawal


'The price for dalchawal is 899'

In [6]:
dish_rate_tool = [
    {
        "type": "function",
        "function": {
            "name": "dish_rate",
            "description": "Returns the price of a dish from the restaurant menu.",
            "parameters": {
                "type": "object",
                "properties": {
                    "dish": {
                        "type": "string",
                        "description": "Name of the dish whose price needs to be checked.",
                    }
                },
                "required": ["dish"],
            },
        },
    }
]

In [7]:
tools = dish_rate_tool

In [11]:
def handle_tool_call(message):
    tool_call = message.tool_calls[0]
    for tool_call in message.tool_calls:
        if tool_call.function.name=="dish_rate":
            arguments= json.loads(tool_call.function.arguments)
            dish = arguments["dish"]
            price_detail = dish_rate(dish)
            response = {
                "role":"tool",
                "content":price_detail,
                "tool_call_id":tool_call.id
            }
        return response


In [ ]:
def chat(message,history):
    history = [{"role":h['role'],"content":h['content']} for h in history]
    messages = [{"role": "system", "content": system_message}]+ history + [{"role": "user", "content": message}]
    response = completion(
        model="groq/llama-3.3-70b-versatile",
        messages=messages,
        tools=tools,
        api_key=groq_api_key,
    )
    while response.choices[0].finish_reason=="tool_calls":
        assistant_message = response.choices[0].message
        tool_response= handle_tool_call(assistant_message)
        messages.append(assistant_message)
        messages.append(tool_response)
        response = completion(
            model="groq/llama-3.3-70b-versatile",
            api_key=groq_api_key,
            messages=messages,
        )

    return response.choices[0].message.content

In [13]:
view = gr.ChatInterface(fn=chat).launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


c:\Users\Sushant\Documents\IntelligenceStack\.venv\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


Menu price check called for butterpaneer


c:\Users\Sushant\Documents\IntelligenceStack\.venv\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
c:\Users\Sushant\Documents\IntelligenceStack\.venv\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


Menu price check called for dalchawal


c:\Users\Sushant\Documents\IntelligenceStack\.venv\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
c:\Users\Sushant\Documents\IntelligenceStack\.venv\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


Menu price check called for pulao


c:\Users\Sushant\Documents\IntelligenceStack\.venv\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
